In [1]:
from transformers import MT5Tokenizer

tokenizer = MT5Tokenizer.from_pretrained("google/mt5-small")


You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [2]:
source = []
with open('en_tran.txt', 'r') as file:
    for line in file:
        source.append(line)
print(source[:10])

['Xiang Ji, Xia Xiangren, named Yu.\n', 'He was twenty-four years old when he started to fight.\n', "His uncle was Xiang Liang, and Xiang Liang's father was Chu general Xiang Yan, who was killed by Qin general Wang Jian.\n", 'The Xiang family has been a general of Chu for generations and was granted the title of Xiang, hence the surname Xiang.\n', 'When Xiang Ji was a child, he learned to read and write but failed.\n', 'He gave up learning calligraphy and switched to fencing, but failed again.\n', 'Xiang Liang was very angry with him.\n', 'Xiang Ji said: Characters are only used to remember names.\n', 'A sword can only withstand one enemy, so it is not worth learning. You must learn to be able to withstand ten thousand people.\n', 'So Xiang Liang taught Xiang Ji the art of war. Xiang Ji was very happy and roughly understood the general idea of \u200b\u200bthe art of war, but he refused to finish it seriously.\n']


In [3]:
lower_limit = 512
upper_limit = 1024

In [4]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

model_name = "sentence-transformers/all-MiniLM-L6-v2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)


def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    mask = attention_mask.unsqueeze(-1).expand(
        token_embeddings.size()
    ).float()
    return (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1)


def sentence_similarity(sen1, sen2):
    encoded = tokenizer(
        [sen1, sen2],
        padding=True,
        truncation=True,
        return_tensors="pt"
    )
    with torch.no_grad():
        output = model(**encoded)
    embeddings = mean_pooling(
        output,
        encoded["attention_mask"]
    )
    similarity = F.cosine_similarity(
        embeddings[0].unsqueeze(0),
        embeddings[1].unsqueeze(0)
    )
    return similarity.item()

def similarity_compare(chunk1, chunk2, sen):
    sim1 = sum([sentence_similarity(s, sen) for s in chunk1]) / len(chunk1)
    sim2 = sum([sentence_similarity(s, sen) for s in chunk2]) / len(chunk2)
    if sim1 >= sim2:
        return 1
    else:
        return 2

In [5]:
chunks = []
current_chunk = []
current_chunk_size = 0
token_source = [len(tokenizer.encode(sen)) for sen in source]

for i,sen in enumerate(source):
    print(f'sentence {i}, chunk {len(chunks)+1}')
    if current_chunk_size + token_source[i] < lower_limit:
        current_chunk.append(sen)
        current_chunk_size += token_source[i]
    elif current_chunk_size + token_source[i] > upper_limit:
        chunks.append(current_chunk)
        current_chunk = []
        current_chunk_size = 0
    else:
        j = i + 1
        next_chunk = []
        next_chunk_size = 0
        while j < len(source):
            if next_chunk_size + token_source[j] > lower_limit:
                break
            next_chunk.append(source[j])
            next_chunk_size += token_source[j]
            j += 1
        if similarity_compare(current_chunk, next_chunk, sen) == 1:
            current_chunk.append(sen)
            current_chunk_size += token_source[i]
        else:
            chunks.append(current_chunk)

            current_chunk = [sen]
            current_chunk_size = token_source[i]
        


sentence 0, chunk 1
sentence 1, chunk 1
sentence 2, chunk 1
sentence 3, chunk 1
sentence 4, chunk 1
sentence 5, chunk 1
sentence 6, chunk 1
sentence 7, chunk 1
sentence 8, chunk 1
sentence 9, chunk 1
sentence 10, chunk 1
sentence 11, chunk 1
sentence 12, chunk 1
sentence 13, chunk 1
sentence 14, chunk 1
sentence 15, chunk 1
sentence 16, chunk 1
sentence 17, chunk 1
sentence 18, chunk 1
sentence 19, chunk 1
sentence 20, chunk 1
sentence 21, chunk 2
sentence 22, chunk 2
sentence 23, chunk 2
sentence 24, chunk 2
sentence 25, chunk 2
sentence 26, chunk 2
sentence 27, chunk 2
sentence 28, chunk 2
sentence 29, chunk 2
sentence 30, chunk 2
sentence 31, chunk 2
sentence 32, chunk 2
sentence 33, chunk 2
sentence 34, chunk 2
sentence 35, chunk 2
sentence 36, chunk 2
sentence 37, chunk 2
sentence 38, chunk 2
sentence 39, chunk 2
sentence 40, chunk 2
sentence 41, chunk 2
sentence 42, chunk 2
sentence 43, chunk 3
sentence 44, chunk 3
sentence 45, chunk 3
sentence 46, chunk 3
sentence 47, chunk 3
se

In [6]:
with open('chunk.txt', 'w') as out:
    for i,chunk in enumerate(chunks):
        out.write(f'Chunk {i}: \n')
        for sen in chunk:
            out.write(sen)
        out.write('\n\n')
        

In [9]:
summary = []
with open('en_sum.txt', 'r') as file:
    for line in file:
        if line.strip():
            summary.append(line)
for s in summary:
    print(s)
print(len(summary))

Xiang Ji, also known as Xiang Yu, was a man from Xiaxiang who began his military career at age twenty-four. His uncle was Xiang Liang, son of Chu general Xiang Yan, who was killed by Qin general Wang Jian. The Xiang family had served as Chu generals for generations and were granted the title of Xiang. As a child, Xiang Ji failed at learning calligraphy and fencing, dismissing them as insufficient for facing ten thousand enemies. He studied the art of war under Xiang Liang but refused to master it completely. After Xiang Liang killed someone, they fled to Wuzhong, where Xiang Liang organized military and funeral services, secretly assessing people's abilities. When Qin Shi Huang visited Kuaiji, Xiang Ji declared he could replace the emperor, prompting Xiang Liang to cover his mouth and warn against such talk. Xiang Liang then recognized Xiang Ji's exceptional nature. Xiang Ji was over eight feet tall, could lift a cauldron, and was feared by the youth of Wuzhong.

In July of Qin II's fi

In [ ]:
from collections import Counter

def rouge_1(orig, tran):
    orig_tokens = orig.lower().split()
    tran_tokens = tran.lower().split()
    orig_count = Counter(orig_tokens)
    tran_count = Counter(tran_tokens)
    combine = orig_count & tran_count
    overlap_count = sum(combine.values())
    precision = overlap_count / len(tran_tokens)
    recall = overlap_count / len(orig_tokens)
    if precision + recall == 0:
        F1 = 0
    else:
        F1 = (2 * precision * recall) / (precision + recall)
    return (precision, recall, F1)

In [12]:
sum_chunk = []
for sen in summary:
    scores = []
    for c in chunks:
        text = ' '.join(c)
        score, _, _ = rouge_1(text, sen)
        scores.append(score)
    idx = scores.index(max(scores))
    sum_chunk.append(idx)

print(sum_chunk)
    

[0, 1, 7, 10, 3]


In [15]:
with open('result.txt', 'w') as out:
    for i,chunk_num in enumerate(sum_chunk):
        out.write(f'Chunk {chunk_num}: \n')
        for sen in chunks[chunk_num]:
            out.write(sen)
        out.write('\n')
        out.write(f'summary line {i}: {summary[i]}')
        out.write('\n\n')